# Recreate reduced NSynth dataset from `.tar.gz` archives, preserving metadata

This notebook rebuilds a reduced NSynth dataset from:

- `nsynth-valid.jsonwav.tar.gz`
- `nsynth-test.jsonwav.tar.gz`

It preserves the annotation metadata stored in `examples.json` and creates a new reduced dataset with:

```text
nsynth/
├── train/
│   ├── *.wav
│   └── examples.json
├── validation/
│   ├── *.wav
│   └── examples.json
└── test/
    ├── *.wav
    └── examples.json
```

The notebook:
1. Mounts Google Drive.
2. Locates the two original NSynth `.tar.gz` archives.
3. Extracts them into a working folder.
4. Loads and merges their metadata.
5. Collects the audio files.
6. Creates a new train / validation / test split.
7. Copies the selected `.wav` files.
8. Writes a matching `examples.json` into each new split.
9. Validates that audio and metadata stay aligned.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import random
import shutil
import tarfile
from collections import Counter

print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


## 1. Configure archive paths and output folders

The default paths below match the folder structure we have been using.  
Only change them if your archives are stored somewhere else.


In [2]:
# Original compressed NSynth splits
VALID_ARCHIVE = Path('/content/drive/MyDrive/TFG_AFTER/data/nsynth-valid.jsonwav.tar.gz')
TEST_ARCHIVE = Path('/content/drive/MyDrive/TFG_AFTER/data/nsynth-test.jsonwav.tar.gz')

# Where the archives will be extracted.
# This folder can be safely recreated.
EXTRACT_ROOT = Path('/content/drive/MyDrive/TFG_AFTER/data/nsynth_extracted_for_rebuild')

# New reduced dataset to create
OUTPUT_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/nsynth')

# Split configuration for the reduced dataset
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

# Reproducibility
SEED = 42

# If True, extracted folders will be recreated.
REEXTRACT_ARCHIVES = False

# If True, the existing reduced train/validation/test folders will be deleted
# before recreating them.
OVERWRITE_EXISTING_SPLITS = True

assert VALID_ARCHIVE.exists(), f"Validation archive not found: {VALID_ARCHIVE}"
assert TEST_ARCHIVE.exists(), f"Test archive not found: {TEST_ARCHIVE}"
assert abs(TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO - 1.0) < 1e-9, "Ratios must sum to 1."

print("Validation archive:", VALID_ARCHIVE)
print("Test archive:", TEST_ARCHIVE)
print("Extraction root:", EXTRACT_ROOT)
print("Output dataset:", OUTPUT_DATASET_ROOT)

Validation archive: /content/drive/MyDrive/TFG_AFTER/data/nsynth-valid.jsonwav.tar.gz
Test archive: /content/drive/MyDrive/TFG_AFTER/data/nsynth-test.jsonwav.tar.gz
Extraction root: /content/drive/MyDrive/TFG_AFTER/data/nsynth_extracted_for_rebuild
Output dataset: /content/drive/MyDrive/datasets/nsynth


## 2. Extract the archives

The archives are only extracted if:
- the expected extracted folders are not found, or
- `REEXTRACT_ARCHIVES = True`.


In [3]:
def extract_archive(archive_path: Path, extract_root: Path) -> None:
    print(f"Extracting: {archive_path.name}")
    with tarfile.open(archive_path, 'r:gz') as tar:
        tar.extractall(path=extract_root)
    print("Done.")

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

expected_valid_dir = EXTRACT_ROOT / 'nsynth-valid'
expected_test_dir = EXTRACT_ROOT / 'nsynth-test'

if REEXTRACT_ARCHIVES:
    if expected_valid_dir.exists():
        shutil.rmtree(expected_valid_dir)
    if expected_test_dir.exists():
        shutil.rmtree(expected_test_dir)

if REEXTRACT_ARCHIVES or not expected_valid_dir.exists():
    extract_archive(VALID_ARCHIVE, EXTRACT_ROOT)
else:
    print("Validation split already extracted:", expected_valid_dir)

if REEXTRACT_ARCHIVES or not expected_test_dir.exists():
    extract_archive(TEST_ARCHIVE, EXTRACT_ROOT)
else:
    print("Test split already extracted:", expected_test_dir)

print("\nExtracted directories:")
print("Validation:", expected_valid_dir, "exists:", expected_valid_dir.exists())
print("Test:", expected_test_dir, "exists:", expected_test_dir.exists())

Extracting: nsynth-valid.jsonwav.tar.gz


/tmp/ipykernel_2790/4050361930.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_root)


Done.
Extracting: nsynth-test.jsonwav.tar.gz
Done.

Extracted directories:
Validation: /content/drive/MyDrive/TFG_AFTER/data/nsynth_extracted_for_rebuild/nsynth-valid exists: True
Test: /content/drive/MyDrive/TFG_AFTER/data/nsynth_extracted_for_rebuild/nsynth-test exists: True


## 3. Check extracted content

Each extracted NSynth folder should contain:
- `examples.json`
- a collection of `.wav` files, usually inside an `audio/` subfolder.


In [4]:
SOURCE_VALIDATION_DIR = expected_valid_dir
SOURCE_TEST_DIR = expected_test_dir

def find_audio_dir(split_dir: Path) -> Path:
    audio_dir = split_dir / 'audio'
    return audio_dir if audio_dir.exists() else split_dir

for source_dir in [SOURCE_VALIDATION_DIR, SOURCE_TEST_DIR]:
    print("\nChecking:", source_dir)
    print("Exists:", source_dir.exists())
    print("examples.json exists:", (source_dir / 'examples.json').exists())
    audio_dir = find_audio_dir(source_dir)
    print("Audio directory used:", audio_dir)
    print("Number of wav files:", len(list(audio_dir.glob('*.wav'))))


Checking: /content/drive/MyDrive/TFG_AFTER/data/nsynth_extracted_for_rebuild/nsynth-valid
Exists: True
examples.json exists: True
Audio directory used: /content/drive/MyDrive/TFG_AFTER/data/nsynth_extracted_for_rebuild/nsynth-valid/audio
Number of wav files: 12678

Checking: /content/drive/MyDrive/TFG_AFTER/data/nsynth_extracted_for_rebuild/nsynth-test
Exists: True
examples.json exists: True
Audio directory used: /content/drive/MyDrive/TFG_AFTER/data/nsynth_extracted_for_rebuild/nsynth-test/audio
Number of wav files: 4096


## 4. Load and merge original metadata

Each key in `examples.json` should match the corresponding audio filename without `.wav`.


In [5]:
def load_metadata(split_dir: Path) -> dict:
    json_path = split_dir / 'examples.json'
    if not json_path.exists():
        raise FileNotFoundError(f"Missing metadata file: {json_path}")
    with open(json_path, 'r') as f:
        return json.load(f)

validation_metadata = load_metadata(SOURCE_VALIDATION_DIR)
test_metadata = load_metadata(SOURCE_TEST_DIR)

all_metadata = {}
all_metadata.update(validation_metadata)
all_metadata.update(test_metadata)

print("Validation metadata entries:", len(validation_metadata))
print("Test metadata entries:", len(test_metadata))
print("Merged metadata entries:", len(all_metadata))

print("\nExample metadata key:")
first_key = next(iter(all_metadata))
print(first_key)
print(all_metadata[first_key])

Validation metadata entries: 12678
Test metadata entries: 4096
Merged metadata entries: 16774

Example metadata key:
keyboard_acoustic_004-060-025
{'note_str': 'keyboard_acoustic_004-060-025', 'sample_rate': 16000, 'qualities_str': ['dark', 'reverb'], 'instrument_source': 0, 'instrument_family_str': 'keyboard', 'instrument_family': 4, 'note': 278915, 'instrument_source_str': 'acoustic', 'qualities': [0, 1, 0, 0, 0, 0, 0, 0, 1, 0], 'pitch': 60, 'instrument_str': 'keyboard_acoustic_004', 'instrument': 327, 'velocity': 25}


## 5. Collect audio files and verify metadata coverage

In [6]:
def collect_wavs(split_dir: Path) -> list[Path]:
    audio_dir = find_audio_dir(split_dir)
    return sorted(audio_dir.glob('*.wav'))

wav_files = collect_wavs(SOURCE_VALIDATION_DIR) + collect_wavs(SOURCE_TEST_DIR)

# Remove duplicated audio names defensively, preserving first occurrence.
unique_wavs_by_stem = {}
for wav_path in wav_files:
    unique_wavs_by_stem.setdefault(wav_path.stem, wav_path)

wav_files = list(unique_wavs_by_stem.values())

missing_metadata = [wav.stem for wav in wav_files if wav.stem not in all_metadata]

print("Total unique wav files found:", len(wav_files))
print("Wavs missing metadata:", len(missing_metadata))

if missing_metadata:
    print("First missing examples:", missing_metadata[:10])
    raise KeyError(
        "Some WAV files do not have corresponding metadata. "
        "Check whether the extracted examples.json files match the extracted WAV files."
    )

print("All wav files have matching metadata.")

Total unique wav files found: 16774
Wavs missing metadata: 0
All wav files have matching metadata.


## 6. Shuffle and split into reduced train / validation / test

This creates a new split from the union of the original NSynth validation and test splits.


In [7]:
random.seed(SEED)
wav_files_shuffled = wav_files.copy()
random.shuffle(wav_files_shuffled)

n_total = len(wav_files_shuffled)
n_train = int(n_total * TRAIN_RATIO)
n_validation = int(n_total * VALIDATION_RATIO)
n_test = n_total - n_train - n_validation

new_splits = {
    'train': wav_files_shuffled[:n_train],
    'validation': wav_files_shuffled[n_train:n_train + n_validation],
    'test': wav_files_shuffled[n_train + n_validation:],
}

print("Total:", n_total)
for split_name, files in new_splits.items():
    print(f"{split_name}: {len(files)}")

Total: 16774
train: 11741
validation: 2516
test: 2517


## 7. Recreate output folders

In [8]:
OUTPUT_DATASET_ROOT.mkdir(parents=True, exist_ok=True)

for split_name in ['train', 'validation', 'test']:
    split_dir = OUTPUT_DATASET_ROOT / split_name

    if OVERWRITE_EXISTING_SPLITS and split_dir.exists():
        shutil.rmtree(split_dir)

    split_dir.mkdir(parents=True, exist_ok=True)

print("Output folders ready.")

Output folders ready.


## 8. Copy WAV files and write matching `examples.json`

For each new split:
- each `.wav` is copied,
- the corresponding metadata entry is copied into that split's `examples.json`.


In [9]:
for split_name, files in new_splits.items():
    split_dir = OUTPUT_DATASET_ROOT / split_name
    split_metadata = {}

    for wav_path in files:
        destination = split_dir / wav_path.name
        shutil.copy2(wav_path, destination)
        split_metadata[wav_path.stem] = all_metadata[wav_path.stem]

    json_path = split_dir / 'examples.json'
    with open(json_path, 'w') as f:
        json.dump(split_metadata, f, indent=2)

    print(
        f"{split_name}: copied {len(files)} wavs and wrote "
        f"{len(split_metadata)} metadata entries -> {json_path}"
    )

train: copied 11741 wavs and wrote 11741 metadata entries -> /content/drive/MyDrive/datasets/nsynth/train/examples.json
validation: copied 2516 wavs and wrote 2516 metadata entries -> /content/drive/MyDrive/datasets/nsynth/validation/examples.json
test: copied 2517 wavs and wrote 2517 metadata entries -> /content/drive/MyDrive/datasets/nsynth/test/examples.json


## 9. Validate the recreated dataset

This checks:
- number of `.wav` files in each split,
- number of metadata entries in each split,
- that every `.wav` has a corresponding metadata key,
- that every metadata key has a corresponding `.wav`.


In [10]:
def validate_split(split_name: str):
    split_dir = OUTPUT_DATASET_ROOT / split_name
    wavs = sorted(split_dir.glob('*.wav'))

    with open(split_dir / 'examples.json', 'r') as f:
        metadata = json.load(f)

    wav_stems = {wav.stem for wav in wavs}
    metadata_keys = set(metadata.keys())

    missing_metadata = wav_stems - metadata_keys
    missing_audio = metadata_keys - wav_stems

    print(f"\nSplit: {split_name}")
    print("Wavs:", len(wavs))
    print("Metadata entries:", len(metadata))
    print("Wavs without metadata:", len(missing_metadata))
    print("Metadata entries without wav:", len(missing_audio))

    if missing_metadata:
        print("Examples missing metadata:", list(missing_metadata)[:10])
    if missing_audio:
        print("Examples missing audio:", list(missing_audio)[:10])

    assert not missing_metadata, f"{split_name}: some WAVs are missing metadata."
    assert not missing_audio, f"{split_name}: some metadata entries are missing WAVs."

for split_name in ['train', 'validation', 'test']:
    validate_split(split_name)

print("\nDataset successfully recreated with aligned metadata.")


Split: train
Wavs: 11741
Metadata entries: 11741
Wavs without metadata: 0
Metadata entries without wav: 0

Split: validation
Wavs: 2516
Metadata entries: 2516
Wavs without metadata: 0
Metadata entries without wav: 0

Split: test
Wavs: 2517
Metadata entries: 2517
Wavs without metadata: 0
Metadata entries without wav: 0

Dataset successfully recreated with aligned metadata.


## 10. Optional: inspect available labels in the recreated dataset

This helps verify that the annotations needed later for Synesis are present.


In [11]:
sample_split = 'train'
with open(OUTPUT_DATASET_ROOT / sample_split / 'examples.json', 'r') as f:
    sample_metadata = json.load(f)

family_counter = Counter()
source_counter = Counter()
qualities_counter = Counter()

for item in sample_metadata.values():
    family_counter[item.get('instrument_family_str', 'MISSING')] += 1
    source_counter[item.get('instrument_source_str', 'MISSING')] += 1

    for quality in item.get('qualities_str', []):
        qualities_counter[quality] += 1

print("Instrument families in train:")
print(family_counter)

print("\nInstrument sources in train:")
print(source_counter)

print("\nMost common qualities in train:")
print(qualities_counter.most_common(15))

Instrument families in train:
Counter({'bass': 2437, 'keyboard': 2221, 'guitar': 1935, 'organ': 1474, 'brass': 838, 'string': 784, 'reed': 654, 'mallet': 580, 'flute': 455, 'vocal': 363})

Instrument sources in train:
Counter({'acoustic': 4773, 'electronic': 4096, 'synthetic': 2872})

Most common qualities in train:
[('reverb', 2731), ('distortion', 2504), ('dark', 2102), ('fast_decay', 1996), ('bright', 1847), ('long_release', 1566), ('nonlinear_env', 1310), ('percussive', 1149), ('multiphonic', 518), ('tempo-synced', 283)]


## 11. Next step

Once this notebook completes successfully, your reduced dataset should be ready for the Synesis notebook.

The folder:

```text
/content/drive/MyDrive/datasets/nsynth
```

will contain:
- audio files,
- split structure,
- and `examples.json` metadata aligned with each new split.
